In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,DraftKings Pick6,player_points,Pascal Siakam,Over,27.5,-137,2025-11-20,2025-11-20T00:50:33Z
1,DraftKings Pick6,player_points,Pascal Siakam,Under,27.5,-137,2025-11-20,2025-11-20T00:50:33Z
2,DraftKings Pick6,player_points,Miles Bridges,Over,23.5,-137,2025-11-20,2025-11-20T00:50:33Z
3,DraftKings Pick6,player_points,Miles Bridges,Under,23.5,-137,2025-11-20,2025-11-20T00:50:33Z
4,DraftKings Pick6,player_points,Kon Knueppel,Over,23.5,-137,2025-11-20,2025-11-20T00:50:33Z


### Update projected starting lineups

In [23]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [18]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 109 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,BetRivers,21.5,25.27,Over,120,0,5.06,0.421,High
1,Dereck Lively II,BetMGM,4.5,7.96,Over,-125,0,4.68,0.585,Low
2,Josh Giddey,BetRivers,20.5,25.27,Over,102,1,4.51,0.442,High
3,Pelle Larsson,BetRivers,10.5,13.30,Over,112,0,4.37,0.390,High
4,Landry Shamet,BetRivers,10.5,13.84,Over,100,0,4.20,0.420,High


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 68 players...
Error getting prediction for Coby White: float division by zero
Processing 63 players with valid predictions...
Generated 1813 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 94 combinations from 1813 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Gui Santos,Dereck Lively II,7.5,4.5,13.56,7.96,over,over,0,8.31,0.416,High,Low
1,Gui Santos,Isaac Okoro,7.5,5.5,13.56,9.04,over,over,0,7.35,0.368,High,High
2,Gui Santos,Landry Shamet,7.5,9.5,13.56,13.84,over,over,0,7.27,0.364,High,High
3,Dereck Lively II,Josh Giddey,4.5,19.5,7.96,25.27,over,over,0,6.85,0.342,Low,High
4,Dereck Lively II,Isaac Okoro,4.5,5.5,7.96,9.04,over,over,0,6.83,0.342,Low,High
5,Landry Shamet,Josh Giddey,9.5,19.5,13.84,25.27,over,over,0,5.54,0.277,High,High
6,Jerami Grant,Isaac Okoro,22.5,5.5,17.27,9.04,under,over,0,5.53,0.276,High,High
7,Josh Giddey,Jerami Grant,19.5,22.5,25.27,17.27,over,under,1,5.50,0.275,High,High
8,Landry Shamet,Jerami Grant,9.5,22.5,13.84,17.27,over,under,0,5.12,0.256,High,High
9,Will Richard,Ayo Dosunmu,16.5,11.5,12.80,14.95,under,over,0,3.51,0.175,High,High


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 70 players...
Error getting prediction for Coby White: float division by zero
Processing 65 players with valid predictions...
Generated 1891 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 95 combinations from 1891 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Zion Williamson,Isaac Okoro,18.5,5.5,24.09,9.04,over,over,0,7.52,0.376,Med,High
1,Zion Williamson,Josh Giddey,18.5,19.5,24.09,25.27,over,over,1,7.12,0.356,Med,High
2,Zion Williamson,Landry Shamet,18.5,9.5,24.09,13.84,over,over,0,7.06,0.353,Med,High
3,Jerami Grant,Isaac Okoro,22.5,5.5,17.27,9.04,under,over,0,5.64,0.282,High,High
4,Landry Shamet,Josh Giddey,9.5,19.5,13.84,25.27,over,over,0,5.60,0.280,High,High


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 68 players...
Error getting prediction for Coby White: float division by zero
Processing 63 players with valid predictions...
Generated 38837 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 41 combinations from 38837 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Gui Santos,Dereck Lively II,Isaac Okoro,7.5,4.5,5.5,13.56,7.96,9.04,over,over,over,0,15.39,0.308,High,Low,High
1,Gui Santos,Dereck Lively II,Josh Giddey,7.5,4.5,19.5,13.56,7.96,25.27,over,over,over,0,15.12,0.302,High,Low,High
2,Landry Shamet,Josh Giddey,Isaac Okoro,9.5,19.5,5.5,13.84,25.27,9.04,over,over,over,0,11.70,0.234,High,High,High
3,Landry Shamet,Jerami Grant,Kevin Huerter,9.5,22.5,10.5,13.84,17.27,14.35,over,under,over,0,9.66,0.193,High,High,High
4,Will Richard,Jerami Grant,Kevin Huerter,16.5,22.5,10.5,12.80,17.27,14.35,under,under,over,0,8.04,0.161,High,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 70 players...
Error getting prediction for Coby White: float division by zero
Processing 65 players with valid predictions...
Generated 42142 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 42 combinations from 42142 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Zion Williamson,Josh Giddey,Isaac Okoro,18.5,19.5,5.5,24.09,25.27,9.04,over,over,over,0,13.88,0.278,Med,High,High
1,Zion Williamson,Landry Shamet,Isaac Okoro,18.5,9.5,5.5,24.09,13.84,9.04,over,over,over,0,13.56,0.271,Med,High,High
2,Landry Shamet,Jerami Grant,Josh Giddey,9.5,22.5,19.5,13.84,17.27,25.27,over,under,over,0,10.92,0.218,High,High,High
3,Dereck Lively II,Jerami Grant,Kevin Huerter,5.5,22.5,10.5,7.96,17.27,14.35,over,under,over,0,9.17,0.183,Low,High,High
4,Dereck Lively II,Ayo Dosunmu,Kevin Huerter,5.5,11.5,10.5,7.96,14.95,14.35,over,over,over,0,8.19,0.164,Low,High,High
